In [1]:
import argparse
import ast
import csv
import json
import sys
import urllib.error
import urllib.request
from typing import Any, Dict, List, Optional


In [2]:

def clean(value: Optional[str], default: str = "") -> str:
    if value is None:
        return default
    v = str(value).strip()
    if v == "" or v.lower() in {"none", "null"}:
        return default
    return v


def to_int(value: Optional[str], default: int = 0) -> int:
    v = clean(value)
    if not v:
        return default
    try:
        return int(float(v))
    except ValueError:
        return default


def to_float(value: Optional[str], default: float = 0.5) -> float:
    v = clean(value)
    if not v:
        return default
    try:
        return float(v)
    except ValueError:
        return default



In [ ]:

def parse_list_field(value: Optional[str]) -> List[str]:
    v = clean(value)
    if not v:
        return []
    try:
        parsed = ast.literal_eval(v)
        if isinstance(parsed, list):
            return [str(x) for x in parsed if str(x).strip()]
    except Exception:
        pass

    # fallback if value is not a valid Python list string
    return [p.strip() for p in v.split(",") if p.strip()]



In [4]:

def map_row_to_alert(row: Dict[str, str]) -> Dict[str, Any]:
    rule_level = to_int(row.get("_source.rule.level"), 3)
    mitre_ids = parse_list_field(row.get("_source.rule.mitre.id"))
    mitre_tactic = parse_list_field(row.get("_source.rule.mitre.tactic"))
    mitre_technique = parse_list_field(row.get("_source.rule.mitre.technique"))

    timestamp = clean(row.get("_source.timestamp")) or clean(row.get("_source.@timestamp"))
    src_ip = clean(row.get("_source.agent.ip"), "unknown")
    agent_name = clean(row.get("_source.agent.name"), "unknown")
    rule_description = clean(row.get("_source.rule.description"), "No description")

    # Optional enrichments
    user = (
        clean(row.get("_source.data.win.eventdata.user"))
        or clean(row.get("_source.data.win.eventdata.targetUserName"))
        or clean(row.get("_source.data.win.eventdata.subjectUserName"))
    )

    groups = parse_list_field(row.get("_source.rule.groups"))
    ml_anomaly_score = max(0.1, min(1.0, rule_level / 15.0))

    return {
        "rule_description": rule_description,
        "src_ip": src_ip,
        "timestamp": timestamp,
        "rule_level": rule_level,
        "ml_severity": "",
        "ml_attack_category": "",
        "ml_anomaly_score": ml_anomaly_score,
        "agent_name": agent_name,
        "extra": {
            "alert_id": clean(row.get("_source.id")) or clean(row.get("_id")) or "csv-row",
            "rule_id": clean(row.get("_source.rule.id")),
            "agent_id": clean(row.get("_source.agent.id")),
            "user": user,
            "mitre_ids": mitre_ids,
            "mitre_tactic": mitre_tactic[0] if mitre_tactic else "",
            "mitre_technique": mitre_technique[0] if mitre_technique else "",
            "groups": groups,
            "full_log": clean(row.get("_source.full_log")),
            "event_id": clean(row.get("_source.data.win.system.eventID")),
            "host": clean(row.get("_source.data.win.system.computer")),
        },
    }


In [5]:

def is_windows_like(row: Dict[str, str]) -> bool:
    groups = [g.lower() for g in parse_list_field(row.get("_source.rule.groups"))]
    location = clean(row.get("_source.location")).lower()
    has_win_payload = any(
        clean(row.get(k))
        for k in [
            "_source.data.win.system.channel",
            "_source.data.win.system.eventID",
            "_source.data.win.eventdata.image",
        ]
    )
    return (
        "windows" in groups
        or "windows_security" in groups
        or location == "eventchannel"
        or has_win_payload
    )



In [6]:

def call_analyze_api(base_url: str, alert_payload: Dict[str, Any]) -> Dict[str, Any]:
    url = base_url.rstrip("/") + "/analyze"
    body = json.dumps(alert_payload).encode("utf-8")
    req = urllib.request.Request(
        url=url,
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        data = resp.read().decode("utf-8")
        return json.loads(data)


In [7]:

def main() -> int:
    parser = argparse.ArgumentParser(description="Map CSV alerts and test SOC agent API")
    parser.add_argument("--csv", default="combined_3000.csv", help="Path to flattened CSV")
    parser.add_argument("--api", default="http://127.0.0.1:8000", help="Agent API base URL")
    parser.add_argument("--limit", type=int, default=5, help="How many alerts to test")
    parser.add_argument("--windows-only", action="store_true", help="Only send windows-like rows")
    parser.add_argument("--dry-run", action="store_true", help="Only print mapped payloads")
    args = parser.parse_args()

    tested = 0
    sent = 0
    failed = 0

    try:
        with open(args.csv, "r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if args.windows_only and not is_windows_like(row):
                    continue

                alert = map_row_to_alert(row)
                tested += 1

                if args.dry_run:
                    print(json.dumps(alert, ensure_ascii=False, indent=2))
                else:
                    try:
                        result = call_analyze_api(args.api, alert)
                        sent += 1
                        report = result.get("report", {})
                        print(
                            f"[OK] {tested} alert_id={alert['extra'].get('alert_id')} "
                            f"severity={report.get('severity', 'n/a')} "
                            f"title={report.get('title', 'n/a')}"
                        )
                    except urllib.error.HTTPError as e:
                        failed += 1
                        err_body = e.read().decode("utf-8", errors="ignore")
                        print(f"[HTTP {e.code}] row={tested} error={err_body}")
                    except Exception as e:
                        failed += 1
                        print(f"[ERROR] row={tested} error={e}")

                if tested >= args.limit:
                    break

    except FileNotFoundError:
        print(f"CSV not found: {args.csv}")
        return 1

    print("\n=== Summary ===")
    print(f"Mapped rows: {tested}")
    print(f"Sent OK:     {sent}")
    print(f"Failed:      {failed}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

usage: ipykernel_launcher.py [-h] [--csv CSV] [--api API] [--limit LIMIT]
                             [--windows-only] [--dry-run]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\PC\AppData\Roaming\jupyter\runtime\kernel-v3b7f80081dac34b3997cf5faf00b3aaeadfca9753.json


SystemExit: 2

c:\Users\PC\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [6]:
_token_cache = {"token": None, "expires_at": 0}

def _get_token() -> str:
    now = time.time()
    if _token_cache["token"] and now < _token_cache["expires_at"]:
        return _token_cache["token"]
    r = requests.post(
        f"{BASE_URL}/security/user/authenticate",
        auth=(WAZUH_USER, WAZUH_PASS),
        verify=False, timeout=10
    )
    r.raise_for_status()
    token = r.json()["data"]["token"]
    _token_cache["token"]      = token
    _token_cache["expires_at"] = now + 840  # 14 min
    return token

def _wazuh_get(endpoint: str, params: dict = None) -> dict:
    headers = {"Authorization": f"Bearer {_get_token()}"}
    r = requests.get(
        f"{BASE_URL}{endpoint}",
        headers=headers, params=params,
        verify=False, timeout=20
    )
    if r.status_code == 200:
        return r.json().get("data", {})
    return {}


In [1]:
import os, requests, json, time
from functools import lru_cache
from langchain_core.tools import tool
from dotenv import load_dotenv
load_dotenv()

WAZUH_HOST = os.getenv("WAZUH_HOST", "192.168.56.30")
WAZUH_PORT = os.getenv("WAZUH_PORT", "55000")
WAZUH_USER = os.getenv("WAZUH_USER", "admin")
WAZUH_PASS = os.getenv("WAZUH_PASSWORD", "")
BASE_URL   = f"https://{WAZUH_HOST}:{WAZUH_PORT}"

# ── JWT token cache (expires after 14 min, Wazuh gives 15) ──────────────────
_token_cache = {"token": None, "expires_at": 0}
def get_user_events(username: str) -> str:
    """
    Query Wazuh to get recent security events involving a specific user
    account. Use this when the alert involves a username (SSH login attempt,
    sudo execution, file access) and you need to understand if this user
    account is behaving normally or suspiciously.

    Args:
        username: the user account name to look up (e.g. 'root', 'testuser')

    Returns:
        JSON string with recent events involving this user.
    """
    try:
        # Query by destination user (auth logs)
        data = _wazuh_get("/alerts", params={
            "limit": 30,
            "sort":  "-timestamp",
            "q":     f"data.dstuser={username}",
        })
        items_dst = data.get("affected_items", [])

        # Also query by source user
        data2 = _wazuh_get("/alerts", params={
            "limit": 30,
            "sort":  "-timestamp",
            "q":     f"data.srcuser={username}",
        })
        items_src = data2.get("affected_items", [])

        all_items = items_dst + items_src

        # Count event types
        failed_logins = sum(
            1 for a in all_items
            if "fail" in a.get("rule", {}).get("description", "").lower()
        )
        sudo_events = sum(
            1 for a in all_items
            if "sudo" in a.get("rule", {}).get("description", "").lower()
        )
        high_severity = sum(
            1 for a in all_items
            if a.get("rule", {}).get("level", 0) >= 10
        )

        recent = []
        for a in all_items[:10]:
            rule = a.get("rule", {})
            recent.append({
                "timestamp":   a.get("timestamp"),
                "description": rule.get("description", ""),
                "level":       rule.get("level"),
                "agent":       a.get("agent", {}).get("name"),
            })

        return json.dumps({
            "username":       username,
            "total_events":   len(all_items),
            "failed_logins":  failed_logins,
            "sudo_events":    sudo_events,
            "high_severity":  high_severity,
            "account_exists": len(all_items) > 0,
            "recent_events":  recent,
        }, ensure_ascii=False)

    except Exception as e:
        return json.dumps({
            "username": username,
            "error":    str(e),
            "total_events": 0
        })


In [5]:
get_user_events("NT AUTHORITY\\LOCAL SERVICE")

'{"username": "NT AUTHORITY\\\\LOCAL SERVICE", "error": "name \'_wazuh_get\' is not defined", "total_events": 0}'

In [2]:

def query_wazuh_logs(ip: str, minutes: int = 15) -> str:
    """
    Query Wazuh API directly to get recent security alerts from a specific
    source IP address. Use this when you need context about what this IP
    has been doing recently (scans, login attempts, file activity).

    Args:
        ip:      source IP address to search (e.g. '192.168.56.10')
        minutes: how far back to look (default 15, max 60)

    Returns:
        JSON string with list of recent alerts from that IP.
    """
    try:
        # Wazuh query syntax: field:value
        data = _wazuh_get("/alerts", params={
            "limit":  50,
            "sort":   "-timestamp",
            "q":      f"data.srcip={ip}",
            "pretty": False,
        })
        items = data.get("affected_items", [])

        # Normalize to useful fields only
        results = []
        for alert in items:
            rule    = alert.get("rule", {})
            results.append({
                "timestamp":   alert.get("timestamp"),
                "rule_id":     str(rule.get("id", "")),
                "rule_level":  rule.get("level"),
                "description": rule.get("description", ""),
                "agent":       alert.get("agent", {}).get("name"),
                "mitre":       rule.get("mitre", {}).get("id", []),
            })

        return json.dumps({
            "ip":     ip,
            "count":  len(results),
            "alerts": results[:20],  # cap at 20 for prompt size
        }, ensure_ascii=False)

    except Exception as e:
        # Graceful fallback — agent continues without this context
        return json.dumps({"ip": ip, "error": str(e), "alerts": []})


In [9]:
get_user_events("mossaabReverse")

c:\Users\PC\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '100.97.198.85'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


'{"username": "mossaabReverse", "error": "401 Client Error: Unauthorized for url: https://100.97.198.85:55000/security/user/authenticate", "total_events": 0}'

In [3]:
query_wazuh_logs("10.0.2.15", 15)

'{"ip": "10.0.2.15", "error": "name \'_wazuh_get\' is not defined", "alerts": []}'

In [4]:
# Patch: use OpenSearch client instead of Wazuh manager JWT auth (avoids 401)

import json

from input.wazuh_client import create_client



def _search_alerts(query: dict, limit: int = 50) -> list[dict]:

    client = create_client()

    body = {

        "size": limit,

        "sort": [{"@timestamp": {"order": "desc"}}],

        "query": query,

    }

    response = client.search(index="wazuh-alerts-*", body=body)

    return response.get("hits", {}).get("hits", [])



def get_user_events(username: str) -> str:

    try:

        items = _search_alerts({

            "bool": {

                "should": [

                    {"term": {"data.dstuser": username}},

                    {"term": {"data.srcuser": username}},

                    {"term": {"data.win.eventdata.user": username}},

                    {"term": {"data.win.eventdata.targetUserName": username}},

                    {"term": {"data.win.eventdata.subjectUserName": username}},

                    {"match": {"full_log": username}},

                ],

                "minimum_should_match": 1,

            }

        }, limit=60)



        all_items = [item.get("_source", {}) for item in items]



        failed_logins = sum(

            1 for a in all_items

            if "fail" in a.get("rule", {}).get("description", "").lower()

        )

        sudo_events = sum(

            1 for a in all_items

            if "sudo" in a.get("rule", {}).get("description", "").lower()

        )

        high_severity = sum(

            1 for a in all_items

            if a.get("rule", {}).get("level", 0) >= 10

        )



        recent = []

        for a in all_items[:10]:

            rule = a.get("rule", {})

            recent.append({

                "timestamp": a.get("timestamp") or a.get("@timestamp"),

                "description": rule.get("description", ""),

                "level": rule.get("level"),

                "agent": a.get("agent", {}).get("name"),

            })



        return json.dumps({

            "username": username,

            "total_events": len(all_items),

            "failed_logins": failed_logins,

            "sudo_events": sudo_events,

            "high_severity": high_severity,

            "account_exists": len(all_items) > 0,

            "recent_events": recent,

        }, ensure_ascii=False)

    except Exception as e:

        return json.dumps({"username": username, "error": str(e), "total_events": 0})



def query_wazuh_logs(ip: str, minutes: int = 15) -> str:

    try:

        items = _search_alerts({

            "bool": {

                "should": [

                    {"term": {"src_ip": ip}},

                    {"term": {"agent.ip": ip}},

                    {"term": {"data.srcip": ip}},

                    {"term": {"data.win.eventdata.sourceIp": ip}},

                    {"match": {"full_log": ip}},

                ],

                "minimum_should_match": 1,

            }

        }, limit=50)



        results = []

        for item in items:

            alert = item.get("_source", {})

            rule = alert.get("rule", {})

            results.append({

                "timestamp": alert.get("timestamp") or alert.get("@timestamp"),

                "rule_id": str(rule.get("id", "")),

                "rule_level": rule.get("level"),

                "description": rule.get("description", ""),

                "agent": alert.get("agent", {}).get("name"),

                "mitre": rule.get("mitre", {}).get("id", []),

            })



        return json.dumps({"ip": ip, "count": len(results), "alerts": results[:20]}, ensure_ascii=False)

    except Exception as e:

        return json.dumps({"ip": ip, "error": str(e), "alerts": []})


In [6]:
query_wazuh_logs("192.168.117.130")

'{"ip": "192.168.117.130", "count": 0, "alerts": []}'

In [7]:
get_user_events("SYSTEM")

'{"username": "SYSTEM", "total_events": 60, "failed_logins": 0, "sudo_events": 0, "high_severity": 53, "account_exists": true, "recent_events": [{"timestamp": "2026-04-26T13:06:25.437+0100", "description": "File deleted.", "level": 7, "agent": "wazuh-VMware-Virtual-Platform"}, {"timestamp": "2026-04-26T13:06:25.437+0100", "description": "File deleted.", "level": 7, "agent": "wazuh-VMware-Virtual-Platform"}, {"timestamp": "2026-04-26T13:06:25.434+0100", "description": "File deleted.", "level": 7, "agent": "wazuh-VMware-Virtual-Platform"}, {"timestamp": "2026-04-26T13:06:21.116+0100", "description": "File added to the system.", "level": 5, "agent": "wazuh-VMware-Virtual-Platform"}, {"timestamp": "2026-04-26T13:06:21.103+0100", "description": "File added to the system.", "level": 5, "agent": "wazuh-VMware-Virtual-Platform"}, {"timestamp": "2026-04-26T13:06:21.080+0100", "description": "File added to the system.", "level": 5, "agent": "wazuh-VMware-Virtual-Platform"}, {"timestamp": "2026-0